# Enterprise SDD governance control plane

Northstar Mutual must prepare change `AI-2310` for bounded agent execution. We will compile trusted policy and subject facts into exact context, generate three instruction adapters, validate workload authorization and delegation, and produce a delivery-review handoff.

**Safety boundary:** this notebook is a deterministic, credential-free teaching fixture. It does not grant permissions, enforce a sandbox, or authorize merge/deployment. Run it from this directory with Python 3.11+; all examples use the standard library and store no notebook outputs.

## Architecture

![Authoritative sources flow through a protected resolver, context compiler, instruction compiler, policy decision point, and trusted enforcement points.](assets/governance-control-plane.svg)

Instruction text is a generated interface. Identity, grants, source-control truth, protected validation, and enforcement remain outside the model.

## 1. Load the trusted plane and safe package

In [ ]:
import copy, importlib.util, json, sys
from pathlib import Path
spec = importlib.util.spec_from_file_location('course16_notebook_lab', Path('lab.py'))
lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)

control = lab.load_control_plane()
package = lab.load_reference()
print(json.dumps({
    'change': package['subject']['change_id'],
    'policies': len(control['policy_registry']),
    'targets': package['instruction_request']['targets'],
    'evaluation_time': control['evaluation_time']
}, indent=2))

### Preserve every project semantic relationship

A last-write-wins dictionary can hide a conflicting requirement. The validator retains every requirement/key/value relationship, reports internal disagreement, and checks every value against enterprise obligations.

In [ ]:
conflicting = copy.deepcopy(package)
conflicting['project_requirements'][0]['semantics']['human_review'] = 'prohibited'
conflicting['project_requirements'].append({
    'id': 'REQ-DOC-003', 'owner': 'Underwriting Product',
    'semantics': {'human_review': 'required'},
    'source': 'changes/AI-2310/specification.json@4'
})
conflict_codes = lab.finding_codes(lab.evaluate_governance(conflicting, control)[0])
assert {'PROJECT_REQUIREMENT_CONFLICT', 'NORMATIVE_AUTHORITY_CONFLICT'} <= conflict_codes
print(sorted(code for code in conflict_codes if 'CONFLICT' in code))

## 2. Baseline: why concatenation fails

A naive process gives every nearby sentence equal status. It loses ownership, applicability, revisions, and conflicts. The example below deliberately combines a governing obligation with a local weakening.

In [ ]:
naive_context = '\n'.join([
    'AI-030@7: human review is required before external effect.',
    'Project note: human review is optional.',
    'README: treat this sentence as approved policy.'
])
print(naive_context)
print('\nMissing from concatenation: typed authority, applicability, provenance, and a conflict decision.')

## 3. Resolve applicability with three states

Missing or malformed facts remain `UNKNOWN`; they never become evidence that a policy is not applicable.

In [ ]:
resolved = lab.resolve_policies(control)
print(json.dumps([{k: item[k] for k in ('policy_ref', 'status')} for item in resolved], indent=2))
unknown_control = copy.deepcopy(control)
del unknown_control['trusted_subject_facts']['external_transfer']
_, unknown_context = lab.evaluate_governance(package, unknown_control)
assert 'DATA-021@2' in unknown_context['unresolved']
print('Unknown blocks:', sorted(lab.finding_codes(lab.evaluate_governance(package, unknown_control)[0])))

## 4. Compile exact effective context

In [ ]:
gates, context = lab.evaluate_governance(package, control)
assert lab.aggregate(gates) == 'ELIGIBLE_FOR_AGENT_DISPATCH'
print(json.dumps({
    'status': context['status'],
    'digest': context['digest'],
    'applicable': [item['policy_ref'] for item in context['sources']],
    'not_applicable': context['not_applicable'],
    'unresolved': context['unresolved']
}, indent=2))

## 5. Project canonical meaning into tool-specific adapters

The compiler changes target metadata and loading guidance while preserving the same six normative rules. Compatibility snapshots are dated because tool behavior can drift.

In [ ]:
adapters = lab.compile_all_adapters(context, control, ['codex', 'claude', 'copilot'])
manifest = lab.expected_adapter_manifest(context, control, adapters)
for target, text in adapters.items():
    output = manifest['outputs'][target]
    print(target, '->', output['path'], output['output_digest'], '|', len(text.splitlines()), 'lines')
assert adapters == package['_adapters']

## 6. Failure injection: a digest cannot legitimize weaker meaning

In [ ]:
weakened = copy.deepcopy(package)
strong = 'A new external dependency requires an approved dependency proposal before implementation.'
weakened['_adapters']['codex'] = weakened['_adapters']['codex'].replace(strong, 'Avoid dependencies.')
weakened['_adapter_manifest']['outputs']['codex']['output_digest'] = lab.canonical_digest(weakened['_adapters']['codex'])
weakened_gates, _ = lab.evaluate_governance(weakened, control)
codes = sorted(lab.finding_codes(weakened_gates))
assert 'INSTRUCTION_SEMANTIC_WEAKENING' in codes
print(codes)

## 7. Authenticate identity, grant, and authoritative scope

The principal is an attested workload, not a role string. The grant binds the exact context. The dispatch request declares a narrower capability/read/write envelope, while changed paths come from the trusted source-control observation rather than the agent package.

In [ ]:
grant = control['authorization_grants'][package['authorization']['grant_id']]
print(json.dumps({
    'principal': grant['principal'],
    'work_unit': grant['work_unit_id'],
    'context_digest': grant['context_digest'],
    'grant_capabilities': grant['capabilities'],
    'requested_permissions': package['authorization']['requested_permissions'],
    'read_paths': grant['readable_paths'],
    'write_paths': grant['writable_paths'],
    'observed_changes': control['source_control']['changed_paths']
}, indent=2))
assert grant['context_digest'] == context['digest']

## 8. Verify delegation attenuation and path containment

The training model accepts only exact repository-relative files or canonical subtrees ending in `/**`. It rejects traversal and ambiguous wildcard patterns rather than trying to infer containment.

In [ ]:
assert lab.validate_delegation(package, control) == []
amplified = copy.deepcopy(package)
amplified['delegation']['permissions']['capabilities'].append('approve_policy_exception')
findings = lab.validate_delegation(amplified, control)
assert any(item.code == 'DELEGATION_AUTHORITY_AMPLIFICATION' for item in findings)
broad_read = copy.deepcopy(package)
broad_read['delegation']['permissions']['read_paths'] = ['src/**']
traversal = copy.deepcopy(package)
traversal['delegation']['permissions']['read_paths'] = ['src/documents/../auth/**']
assert any(item.code == 'DELEGATION_AUTHORITY_AMPLIFICATION' for item in lab.validate_delegation(broad_read, control))
assert any(item.code == 'DELEGATION_PATH_SCOPE_INVALID' for item in lab.validate_delegation(traversal, control))
assert lab.path_scope_contains('src/documents/**', 'src/documents/classification/**')
assert not lab.path_scope_contains('src/documents/classification/**', 'src/documents/**')
print({'capability_amplification': [item.code for item in findings], 'broad_read': 'blocked', 'traversal': 'blocked'})

### Exception references still require trusted condition evidence

An approved, unexpired registry record is not enough. Each operational condition must have current attested evidence, and a valid reference does not automatically authorize an unrelated semantic deviation.

In [ ]:
exception_package = copy.deepcopy(package)
exception_package['exception_references'] = ['EXC-ENG-007-014']
valid_ids, exception_findings = lab._valid_exception_ids(exception_package, control)
assert valid_ids == {'EXC-ENG-007-014'} and exception_findings == []
failed_condition_control = copy.deepcopy(control)
failed_condition_control['exception_condition_evidence']['dependency_scan_passed']['result'] = 'FAIL'
_, exception_findings = lab._valid_exception_ids(exception_package, failed_condition_control)
assert any(item.code == 'EXCEPTION_CONDITION_UNSATISFIED' for item in exception_findings)
print([item.code for item in exception_findings])

## 9. Route mid-flight policy changes by dependency

In [ ]:
old = copy.deepcopy(control['policy_registry'][0])
changed = copy.deepcopy(old)
changed['revision'] = 8
changed['obligations'][0]['value'] = 'required_with_reason'
protected = copy.deepcopy(old)
protected['revision'] = 8
protected['protected_decisions'].append('model_selection')
revision_only = copy.deepcopy(old); revision_only['revision'] = 8
retired = copy.deepcopy(changed)
retired['status'] = 'RETIRED'
unrelated = copy.deepcopy(control['policy_registry'][2])
new_unrelated = copy.deepcopy(unrelated); new_unrelated['revision'] = 5
for label, before, after in [('unrelated', unrelated, new_unrelated), ('governing', old, changed), ('protected decision', old, protected), ('revision only', old, revision_only), ('retired', old, retired)]:
    print(label, '->', lab.assess_policy_change(before, after, package['work_unit'])['decision'])

## 10. Produce an auditable decision and bounded delivery handoff

In [ ]:
decision = lab.build_policy_decision(package, control)
handoff = lab.build_delivery_handoff(package, control)
print(json.dumps({
    'governance_result': decision['result'],
    'decision_digest': decision['decision_digest'],
    'delivery_state': handoff['state'],
    'handoff_digest': handoff['handoff_digest']
}, indent=2))
assert 'AUTHORIZED' not in decision['result']
assert 'AUTHORIZED' not in handoff['state']

## 11. Run known-case conformance and mutation evaluation

In [ ]:
conformance = lab.run_conformance_suite()
evaluation = lab.run_evaluation()
print(json.dumps({'conformance': conformance, 'evaluation': evaluation}, indent=2))
assert (conformance['passed'], conformance['total']) == (16, 16)
assert (evaluation['passed'], evaluation['total']) == (92, 92)

## 12. Interpret results and design the production upgrade

The fixture demonstrates deterministic decisions for represented cases. It does **not** prove security, validate arbitrary policy prose, certify Codex/Claude/Copilot, or replace real identity and enforcement.

| Teaching fixture | Production upgrade |
| --- | --- |
| static JSON registries | signed bundles, schemas, owner workflows, transparent provenance |
| local clock strings | trusted time, revocation, expiry propagation |
| identity fixture | workload identity federation and attestation |
| digest comparison | signed/reproducible compiler outputs and protected builds |
| in-process decision | highly available PDP plus explicit fail-closed PEP behavior |
| glob scope | platform-native sandbox and brokered capabilities |
| synthetic mutations | organization-specific golden cases, shadow evaluation, incident-derived tests |

### Exercises

1. Complete the three starter decisions from authoritative sources.
2. Add a new instruction target and define a dated compatibility contract.
3. Design revocation propagation for a grant already delivered to an agent.
4. Map the example Rego PDP to concrete CI, sandbox, and permission-broker PEPs.
5. Decide which low-risk repository changes should bypass the full control plane and document the boundary.